In [ ]:
!pip -q install -U transformers accelerate sentencepiece huggingface_hub


In [ ]:
import json
import re
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B-it"

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, switch to a T4 GPU runtime before running this notebook.")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print(f"Loaded {MODEL_ID}")
print(f"CUDA device: {torch.cuda.get_device_name(0)}")


In [ ]:
def clean_final_answer(text):

    text = re.sub(r"<turn\|>|<eos>|<bos>", "", text)
    text = re.sub(r"<\|/?[^>]+\|>|<[^>]+>", "", text)
    return text.strip()


def split_gemma_response(raw_text):

    patterns = [
        r"<\|channel\>thought\n(?P<thought>.*?)<channel\|>(?P<answer>.*?)(?:<turn\|>|<eos>|$)",
        r"<start_of_turn>thought\n(?P<thought>.*?)<end_of_turn>(?P<answer>.*?)(?:<end_of_turn>|<eos>|$)",
        r"<think>(?P<thought>.*?)</think>(?P<answer>.*?)(?:<eos>|$)",
    ]

    for pattern in patterns:
        match = re.search(pattern, raw_text, flags=re.DOTALL)
        if match:
            return match.group("thought").strip(), clean_final_answer(match.group("answer")), None

    parsed = None

    try:
        parsed = processor.parse_response(raw_text)
    except Exception:
        parsed = None

    if isinstance(parsed, dict):
        reasoning = parsed.get("thought") or parsed.get("thinking") or parsed.get("reasoning") or ""
        final_answer = parsed.get("answer") or parsed.get("final") or parsed.get("response") or ""
        if reasoning or final_answer:
            return str(reasoning).strip(), clean_final_answer(str(final_answer)), parsed

    if isinstance(parsed, (list, tuple)) and len(parsed) >= 2:
        return str(parsed[0]).strip(), clean_final_answer(str(parsed[1])), parsed

    return "", clean_final_answer(raw_text), parsed


@torch.inference_mode()
def ask_gemma(question, max_new_tokens=4096, do_sample=False, temperature=1.0, top_p=0.95, top_k=64):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": question},
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )

    inputs = processor(text=prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    turn_token_id = processor.tokenizer.convert_tokens_to_ids("<turn|>")

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        eos_token_id=[
            processor.tokenizer.eos_token_id,
            turn_token_id,
        ],
        pad_token_id=processor.tokenizer.eos_token_id,
    )

    if do_sample:
        generation_kwargs.update(
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
        )

    outputs = model.generate(**generation_kwargs)

    generated_ids = outputs[0][input_len:].tolist()
    raw_generation = processor.decode(generated_ids, skip_special_tokens=False)
    cot, final_answer, parsed = split_gemma_response(raw_generation)
    raw_output_tokens = raw_generation

    print("Input Question:")
    print()
    print(question)
    print()
    print("CoT:")
    print()
    print(cot if cot else "[No CoT block could be extracted from the raw generation]")
    print()
    print("Final Answer:")
    print()
    print(final_answer)
    print()
    print("============")
    print("Raw output tokens:")
    print()
    print(raw_output_tokens)

    return {
        "question": question,
        "cot": cot,
        "final_answer": final_answer,
        "raw_generation": raw_generation,
        "generated_token_ids": generated_ids,
        "raw_output_tokens": raw_output_tokens,
        "parsed": parsed,
    }


In [ ]:
import os
import csv
import time
from datetime import datetime, timezone
from pathlib import Path

DATASET_PATH = "/content/pilot_question_pairs_50.jsonl"


def in_colab():
    try:
        import google.colab              
        return True
    except Exception:
        return False


def maybe_upload_dataset(dataset_path=DATASET_PATH):
    dataset_path = Path(dataset_path)
    if dataset_path.exists():
        print(f"Found dataset: {dataset_path}")
        return str(dataset_path)

    if not in_colab():
        raise FileNotFoundError(
            f"Dataset not found at {dataset_path}. Put pilot_question_pairs_50.jsonl there or update DATASET_PATH."
        )

    from google.colab import files
    print("Upload pilot_question_pairs_50.jsonl")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")

    uploaded_name = next(iter(uploaded.keys()))
    target = Path("/content") / uploaded_name
    if target != dataset_path:
        dataset_path.write_bytes(target.read_bytes())
        print(f"Copied uploaded file to {dataset_path}")
    return str(dataset_path)


def load_question_pairs(dataset_path=DATASET_PATH):
    dataset_path = maybe_upload_dataset(dataset_path)
    rows = []
    with open(dataset_path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["_dataset_line"] = line_number
            rows.append(row)

    ids = [row["id"] for row in rows]
    if len(ids) != len(set(ids)):
        raise ValueError("Dataset contains duplicate ids.")

    print(f"Loaded {len(rows)} question pairs from {dataset_path}")
    return rows


question_pairs = load_question_pairs(DATASET_PATH)
question_pairs[:2]


In [ ]:

def build_gemma_prompt(question, system_prompt="You are a helpful assistant."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )


def normalize_injected_cot(injected_cot, injection_is_raw=False):
    injected_cot = injected_cot or ""
    if injection_is_raw:
        return injected_cot
    if injected_cot.lstrip().startswith("<|channel>thought"):
        return injected_cot
    return "<|channel>thought\n" + injected_cot


OUTPUT_DIR = Path("/content/gemma_benchmark_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INJECTION_MODES = {
    "cot_20_percent": 0.20,
    "cot_50_percent": 0.50,
    "cot_90_percent": 0.90,
    "full_cot_answer_now": 1.00,
}

RESULT_BASE_FIELDS = [
    "id", "category", "difficulty", "answer_type", "pair_change_description",
    "expected_solution_method", "method_stability", "original_answer", "counterfactual_answer",
]


def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def gemma_stop_token_ids():
    turn_token_id = processor.tokenizer.convert_tokens_to_ids("<turn|>")
    stop_ids = [processor.tokenizer.eos_token_id]
    if isinstance(turn_token_id, int) and turn_token_id >= 0:
        stop_ids.append(turn_token_id)
    return stop_ids


@torch.inference_mode()
def generate_gemma_raw(question, max_new_tokens=4096, do_sample=False, temperature=1.0, top_p=0.95, top_k=64):
    prompt = build_gemma_prompt(question)
    inputs = processor(text=prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        eos_token_id=gemma_stop_token_ids(),
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)

    outputs = model.generate(**generation_kwargs)
    generated_ids = outputs[0][input_len:].tolist()
    raw_generation = processor.decode(generated_ids, skip_special_tokens=False)
    cot, final_answer, parsed = split_gemma_response(raw_generation)
    return {
        "cot": cot,
        "final_answer": final_answer,
        "raw_generation": raw_generation,
        "generated_token_ids": generated_ids,
        "parsed": parsed,
    }


@torch.inference_mode()
def generate_gemma_from_raw_prefix(question, raw_prefix, max_new_tokens=2048, do_sample=False, temperature=1.0, top_p=0.95, top_k=64):
    prompt = build_gemma_prompt(question)
    inputs = processor(text=prompt + raw_prefix, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        eos_token_id=gemma_stop_token_ids(),
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)

    outputs = model.generate(**generation_kwargs)
    continuation_ids = outputs[0][input_len:].tolist()
    continuation = processor.decode(continuation_ids, skip_special_tokens=False)
    raw_generation = raw_prefix + continuation
    cot, final_answer, parsed = split_gemma_response(raw_generation)
    return {
        "cot": cot,
        "final_answer": final_answer,
        "raw_generation": raw_generation,
        "generated_continuation": continuation,
        "generated_token_ids": continuation_ids,
        "parsed": parsed,
    }


def extract_raw_thought_text(raw_generation):
    match = re.search(r"<\|channel\>thought\n(?P<thought>.*?)(?:<channel\|>|<turn\|>|<eos>|$)", raw_generation, flags=re.DOTALL)
    if match:
        return match.group("thought").strip()
    cot, _, _ = split_gemma_response(raw_generation)
    return cot.strip()


def nearest_cut_at_boundary(text, target_ratio):
    text = text.strip()
    if not text:
        return ""
    target = max(1, min(len(text), int(len(text) * target_ratio)))

    boundary_positions = set()
    for match in re.finditer(r"\n\s*\n|(?<=[.!?])\s+|(?<=\))\s+|(?<=:)\s+", text):
        boundary_positions.add(match.end())
    for match in re.finditer(r"\n", text):
        boundary_positions.add(match.end())

    if not boundary_positions:
        return text[:target].rstrip()

    lower = int(len(text) * max(0.05, target_ratio - 0.12))
    upper = int(len(text) * min(0.98, target_ratio + 0.12))
    candidates = [pos for pos in boundary_positions if lower <= pos <= upper]
    if not candidates:
        candidates = list(boundary_positions)

    cut = min(candidates, key=lambda pos: abs(pos - target))
    return text[:cut].rstrip()


def make_counterfactual_cot_prefix(counterfactual_raw_generation, injection_mode="cot_50_percent"):
    if injection_mode not in INJECTION_MODES:
        raise ValueError(f"Unknown injection_mode {injection_mode!r}. Choose one of {list(INJECTION_MODES)}")

    thought = extract_raw_thought_text(counterfactual_raw_generation)
    if not thought:
        raise ValueError("Could not extract a thought/CoT block from the counterfactual raw_generation.")

    if injection_mode == "full_cot_answer_now":
        injected_thought = thought.strip()
        raw_prefix = "<|channel>thought\n" + injected_thought + "<channel|>"
        continuation_instruction = "answer_channel_started"
    else:
        injected_thought = nearest_cut_at_boundary(thought, INJECTION_MODES[injection_mode])
        raw_prefix = "<|channel>thought\n" + injected_thought
        continuation_instruction = "continue_thinking_from_prefix"

    return {
        "injection_mode": injection_mode,
        "target_ratio": INJECTION_MODES[injection_mode],
        "continuation_instruction": continuation_instruction,
        "injected_thought": injected_thought,
        "injected_raw_prefix": raw_prefix,
        "injected_char_count": len(injected_thought),
        "source_thought_char_count": len(thought),
        "actual_char_ratio": round(len(injected_thought) / max(1, len(thought)), 4),
    }


def make_result_record(item, run_kind, question, generation, extra=None):
    record = {field: item.get(field) for field in RESULT_BASE_FIELDS}
    record.update({
        "run_kind": run_kind,
        "question": question,
        "original_question": item["original_question"],
        "counterfactual_question": item["counterfactual_question"],
        "cot": generation["cot"],
        "final_answer": generation["final_answer"],
        "raw_generation": generation["raw_generation"],
        "generated_token_ids": generation.get("generated_token_ids", []),
        "model_id": MODEL_ID,
        "created_at_utc": utc_now_iso(),
    })
    if extra:
        record.update(extra)
    return record


def save_records(records, output_prefix, download=True):
    output_prefix = str(output_prefix)
    jsonl_path = OUTPUT_DIR / f"{output_prefix}.jsonl"
    pretty_path = OUTPUT_DIR / f"{output_prefix}.pretty.json"
    csv_path = OUTPUT_DIR / f"{output_prefix}.csv"

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    with open(pretty_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    all_keys = []
    for record in records:
        for key in record.keys():
            if key not in all_keys:
                all_keys.append(key)
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=all_keys)
        writer.writeheader()
        for record in records:
            row = {}
            for key in all_keys:
                value = record.get(key, "")
                if isinstance(value, (dict, list)):
                    value = json.dumps(value, ensure_ascii=False)
                row[key] = value
            writer.writerow(row)

    print(f"Saved {len(records)} records:")
    print(f"- {jsonl_path}")
    print(f"- {pretty_path}")
    print(f"- {csv_path}")

    if download and in_colab():
        from google.colab import files
        for path in [jsonl_path, pretty_path, csv_path]:
            files.download(str(path))

    return {"jsonl": str(jsonl_path), "pretty_json": str(pretty_path), "csv": str(csv_path)}


def load_records_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


In [ ]:
LIMIT = 10                               
START_INDEX = 0                
MAX_NEW_TOKENS_COUNTERFACTUAL = 4096
DO_SAMPLE = False
DOWNLOAD_RESULTS = True


def run_counterfactual_batch(
    question_pairs,
    limit=LIMIT,
    start_index=START_INDEX,
    max_new_tokens=MAX_NEW_TOKENS_COUNTERFACTUAL,
    do_sample=DO_SAMPLE,
    download=DOWNLOAD_RESULTS,
    output_prefix=None,
):
    selected = question_pairs[start_index: None if limit is None else start_index + limit]
    if output_prefix is None:
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_prefix = f"counterfactual_duplicate_runs_{len(selected)}_{stamp}"

    records = []
    for index, item in enumerate(selected, start=1):
        print(f"[{index}/{len(selected)}] Counterfactual {item['id']}: {item['counterfactual_question'][:100]}")
        generation = generate_gemma_raw(
            item["counterfactual_question"],
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
        )
        records.append(make_result_record(
            item=item,
            run_kind="counterfactual_duplicate",
            question=item["counterfactual_question"],
            generation=generation,
        ))
        print(f"    Final answer: {generation['final_answer'][:200]!r}")

    paths = save_records(records, output_prefix=output_prefix, download=download)
    return records, paths


counterfactual_results, counterfactual_paths = run_counterfactual_batch(question_pairs)
counterfactual_paths


In [ ]:
INJECTION_MODE = "cot_50_percent"                                                                            
MAX_NEW_TOKENS_ORIGINAL = 2048
DOWNLOAD_RESULTS = True


def run_original_with_counterfactual_cot_batch(
    question_pairs,
    counterfactual_results,
    injection_mode=INJECTION_MODE,
    limit=LIMIT,
    start_index=START_INDEX,
    max_new_tokens=MAX_NEW_TOKENS_ORIGINAL,
    do_sample=DO_SAMPLE,
    download=DOWNLOAD_RESULTS,
    output_prefix=None,
):
    selected = question_pairs[start_index: None if limit is None else start_index + limit]
    counterfactual_by_id = {record["id"]: record for record in counterfactual_results}

    missing = [item["id"] for item in selected if item["id"] not in counterfactual_by_id]
    if missing:
        raise ValueError(f"Missing counterfactual results for ids: {missing[:10]}")

    if output_prefix is None:
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_prefix = f"original_with_{injection_mode}_{len(selected)}_{stamp}"

    records = []
    for index, item in enumerate(selected, start=1):
        cf_record = counterfactual_by_id[item["id"]]
        prefix_info = make_counterfactual_cot_prefix(cf_record["raw_generation"], injection_mode=injection_mode)

        print(
            f"[{index}/{len(selected)}] Original {item['id']} with {injection_mode} "
            f"(actual ratio {prefix_info['actual_char_ratio']})"
        )
        generation = generate_gemma_from_raw_prefix(
            item["original_question"],
            raw_prefix=prefix_info["injected_raw_prefix"],
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
        )

        records.append(make_result_record(
            item=item,
            run_kind="original_with_counterfactual_cot_injection",
            question=item["original_question"],
            generation=generation,
            extra={
                **prefix_info,
                "counterfactual_cot": cf_record.get("cot", ""),
                "counterfactual_raw_generation": cf_record.get("raw_generation", ""),
                "counterfactual_model_final_answer": cf_record.get("final_answer", ""),
                "generated_continuation": generation.get("generated_continuation", ""),
            },
        ))
        print(f"    Final answer: {generation['final_answer'][:200]!r}")

    paths = save_records(records, output_prefix=output_prefix, download=download)
    return records, paths


original_injection_results, original_injection_paths = run_original_with_counterfactual_cot_batch(
    question_pairs,
    counterfactual_results,
    injection_mode=INJECTION_MODE,
)
original_injection_paths


In [ ]:
RUN_ALL_MODES = False

if RUN_ALL_MODES:
    all_mode_outputs = {}
    for mode in ["cot_20_percent", "cot_50_percent", "cot_90_percent", "full_cot_answer_now"]:
        print(f"\n===== Running injection mode: {mode} =====")
        records, paths = run_original_with_counterfactual_cot_batch(
            question_pairs,
            counterfactual_results,
            injection_mode=mode,
            limit=LIMIT,
            start_index=START_INDEX,
            max_new_tokens=MAX_NEW_TOKENS_ORIGINAL,
            do_sample=DO_SAMPLE,
            download=DOWNLOAD_RESULTS,
        )
        all_mode_outputs[mode] = {"records": records, "paths": paths}

    all_mode_outputs.keys()
else:
    print("RUN_ALL_MODES is False. Set it to True if you want to run all four injection settings.")


In [ ]:
def show_result(records, index=0, view="summary"):
    record = records[index]
    print(f"id: {record['id']}")
    print(f"run_kind: {record['run_kind']}")
    print(f"category: {record['category']} | difficulty: {record['difficulty']}")
    print(f"original_answer: {record['original_answer']} | counterfactual_answer: {record['counterfactual_answer']}")
    print()

    if view in {"summary", "question"}:
        print("Question:")
        print(record["question"])
        print()
    if view in {"summary", "cot"}:
        print("CoT:")
        print(record.get("cot", ""))
        print()
    if view in {"summary", "final"}:
        print("Final answer:")
        print(record.get("final_answer", ""))
        print()
    if view == "raw":
        print("Raw generation:")
        print(record.get("raw_generation", ""))
    if view == "prefix":
        print("Injected raw prefix:")
        print(record.get("injected_raw_prefix", ""))


           
                                                    
                                                           
                                                            
show_result(original_injection_results, 0, view="summary")


In [ ]:
from threading import Thread
from transformers import TextIteratorStreamer
from IPython.display import display, Markdown


def ask_gemma_continue_streaming(
    question,
    manual_injected_cot_or_raw="",
    injection_is_raw=False,
    max_new_tokens=1024,
    do_sample=False,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
):
    raw_prefix = normalize_injected_cot(manual_injected_cot_or_raw, injection_is_raw=injection_is_raw) if manual_injected_cot_or_raw else ""
    prompt = build_gemma_prompt(question)
    inputs = processor(text=prompt + raw_prefix, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(
        processor.tokenizer,
        skip_prompt=True,
        skip_special_tokens=False,
    )

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        eos_token_id=gemma_stop_token_ids(),
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    print("Input Question:")
    print(question)
    print("\nStreaming raw output tokens:\n")
    print(raw_prefix, end="", flush=True)

    continuation = ""
    for chunk in streamer:
        continuation += chunk
        print(chunk, end="", flush=True)
    thread.join()

    raw_generation = raw_prefix + continuation
    cot, final_answer, parsed = split_gemma_response(raw_generation)

    print("\n\n============")
    print("CoT:\n")
    print(cot)
    print("\nFinal Answer:\n")
    print(final_answer)
    print("\n============")
    print("Raw output tokens:\n")
    print(raw_generation)

    return {
        "question": question,
        "injected_raw_prefix": raw_prefix,
        "generated_continuation": continuation,
        "raw_generation": raw_generation,
        "cot": cot,
        "final_answer": final_answer,
        "parsed": parsed,
    }


In [ ]:
manual_question = "What is 5+5?"
manual_injected_cot_or_raw = """
""".strip()

manual_stream_result = ask_gemma_continue_streaming(
    question=manual_question,
    manual_injected_cot_or_raw=manual_injected_cot_or_raw,
    injection_is_raw=False,
    max_new_tokens=512,
    do_sample=False,
)
